In [ ]:
import fitz
import  unicodedata
import re
import pandas as pd

In [ ]:
def strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", text)
                   if unicodedata.category(ch) != "Mn")

In [ ]:
def extract_blocks(pdf_path: str) -> list[str]:
    doc = fitz.open(pdf_path)
    full = ""
    for page in doc:
        full += page.get_text("text") + "\n\f\n"
    full = strip_accents(full)
    return full.split("FICHA TECNICA")[1:]


In [ ]:
def parse_block(block: str)-> dict | None:
    lines = [ln.strip() for ln in strip_accents(block).splitlines() if ln.strip()]
    upper = [ln.upper() for ln in lines]
    
    #saltar encabezado de columnas si existe
    headers = ["PROYECTO", "RADICACION", "TIPO DE PROYECTO", "SEUDONIMO"]
    if len(upper)>=4 and all (upper[i]==headers[i] for i in range(4)):
        lines = lines[4:]; upper = upper[4:]
    
    rec = {}
    
    # — No. CÁMARA con posible “ACU AL”
    if "NO. CAMARA" in upper:
        i = upper.index("NO. CAMARA")
        if i+1 < len(lines):
            cam_line = lines[i+1]
            # Caso especial ACU AL:
            if "ACU AL" in cam_line.upper():
                left, right = re.split(r"ACU AL", cam_line, flags=re.IGNORECASE)
                m1 = re.search(r"(\d+)/\d{4}[CS]", left)
                m2 = re.search(r"(\d+)/\d{4}[CS]", right)
                if m1 and m2:
                    rec["num_camara"] = m1.group(1)
                    rec["acu_al"]     = m2.group(1)
            else:
                m = re.search(r"(\d+)/\d{4}[CS]", cam_line)
                if m:
                    rec["num_camara"] = m.group(1)

    # — No. SENADO (si existe)
    if "NO. SENADO" in upper:
        j = upper.index("NO. SENADO")
        if j+1 < len(lines):
            sen_line = lines[j+1]
            m = re.search(r"(\d+)/\d{4}[CS]", sen_line)
            if m:
                rec["num_senado"] = m.group(1)

                